# Backtesting against 2026 Events

Now I would like to check the validity of this method against the actual events and boulders held in 2026

In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score, mean_absolute_error

In [2]:


events_2026 = ["KEQ26", "BER26", "MAD26", "PRA26", "IBK26"]
style_labs = ["Slab", "Coordination", "Power", "Compression", "Dynamic", "Press"]

style = pd.read_csv("BoulderStyle.csv")
style["Boulder_Num"] = style["Boulder_Num"].astype(int)

affinity = pd.read_csv("Athlete_Style_Affinity_Shrunk.csv")
affinity_cols = [f"{s}_Affinity_Shrunk" for s in style_labs]

merged = pd.read_csv("Boulder_Results_with_Style.csv")
MLdata = merged.merge(affinity, on="Athlete_ID", how="inner")

for s in style_labs:
    MLdata[f"{s}_Interaction"] = MLdata[s] * MLdata[f"{s}_Affinity_Shrunk"]
interaction_cols = [f"{s}_Interaction" for s in style_labs]
stats_cols = style_labs + affinity_cols + interaction_cols

train_df = MLdata[~MLdata["Event_ID"].isin(events_2026)].copy()

from sklearn.linear_model import Ridge
model = Ridge(alpha=1.0)
model.fit(train_df[stats_cols], train_df["Pct_Of_Best"])

Semifinals = pd.read_csv("SemiFinalists.csv")

roster = Semifinals["Athlete_ID"].tolist()
BoulderResults = pd.read_csv("BoulderResults.csv")

train_df["Predicted_Score"] = model.predict(train_df[stats_cols])
train_df["Residual"] = train_df["Pct_Of_Best"] - train_df["Predicted_Score"]
residual_pool = train_df["Residual"].values

FileNotFoundError: [Errno 2] No such file or directory: 'BoulderStyle.csv'

In [ ]:
# From Book 4

# -----------------------------------------------------------------------
# Generate the boulders (resample new rounds each time)

def generate_simulated_boulders(round_name, num_boulders = 4):
  pool = style[style["Round"] == round_name]       # Pool from previous boulders in that particular round
  sample = pool.sample(n = num_boulders, replace = True).reset_index(drop=True) # Allow for a boulder with the same styles (not uncommon to have multiple coordination boulders)
  return sample[style_labs]

# -----------------------------------------------------------------------
# Now predict the athletes score against the simulated boulders
def score_pred_on_boulders(roster, boulders_df, affinity_df):

  rows = [] # Dictionary to collect the athlete boulder pairings

  for boulder_idx, boulder in boulders_df.reset_index(drop = True).iterrows():
    # Loop for each boulder, and boulder_idx stores the boulder number.
    for athlete_id in roster:
      row = {"Athlete_ID": athlete_id, "Boulder_Idx": boulder_idx}

      for s in style_labs:
        row[s] = boulder[s]
      rows.append(row)

  sim_df = pd.DataFrame(rows)
  # Data frame containing all the athletes and boulders

  sim_df = sim_df.merge(affinity_df, on="Athlete_ID", how="left")
  # Add on the affinity ratings

  for s in style_labs:
    sim_df[f"{s}_Interaction"] = sim_df[s] * sim_df[f"{s}_Affinity_Shrunk"]
  # Again I require the interaction terms for the model

  sim_df["Predicted_Score"] = model.predict(sim_df[stats_cols])
  # Predict the score for each athlete

  noise = np.random.choice(residual_pool, size = len(sim_df))
  # Introducing the random noise that we established before.

  sim_df["Simulated_Score"] = np.clip(sim_df["Predicted_Score"] + noise, 0 ,1)
  # So we have our simulated score including the random noise. Ensuring we cannot leave 0 or 1.

  return sim_df


  # ---------------------------------------------------------------------------
  # Now to rank the round based on the scores

def ranks(sim_df):
  total_score = sim_df.groupby("Athlete_ID")["Simulated_Score"].sum().reset_index()
  # Group rows by athlete and sum the score together from the 4 boulders

  return total_score.sort_values("Simulated_Score", ascending=False).reset_index(drop = True)

  # --------------------------------------------------------------------------

In [ ]:
# Rather than resampling the boulders, we already have the exact boulders used.
# The only sort of uncertainty would come from athlete performance.
# So now the aim is to check the validity of our model against real podiums

def get_real_boulders(event_id, round_name):
  # Get the actual boulders from that event since we no longer have uncertainty for how each boulder will be set
  boulders = style[(style["Event_ID"] == event_id) & (style["Round"] == round_name)]
  return boulders[style_labs].reset_index(drop = True)

def one_trial_real_boulders(roster, sf_boulders, f_boulders, affinity, n_finalists = 8, n_podium = 3):
  # Function to run a trial for each boulder, with the uncertainty fully coming from the athlete's
  # individual performances
  sf_scores = score_pred_on_boulders(roster, sf_boulders, affinity)
  sf_ranking = ranks(sf_scores)
  finalists = sf_ranking.head(n_finalists)["Athlete_ID"].tolist()

  f_scores = score_pred_on_boulders(finalists, f_boulders, affinity)
  f_ranking = ranks(f_scores)
  podium = f_ranking.head(n_podium)["Athlete_ID"].tolist()

  return finalists, podium

def podium_for_event(event_id, results_df):
  # Obtain the podium results for the event
  final_rows = results_df[(results_df["Event_ID"] == event_id) & (results_df["Round"] == "Final")]
  totals = final_rows.groupby("Athlete_ID")["Score"].sum().reset_index()
  totals = totals.sort_values("Score", ascending = False).reset_index(drop = True)
  return totals.head(3)["Athlete_ID"].tolist()

def actual_finalists_for_event(event_id, results_df):
    # Obtain the real finalists results for the event
    return set(results_df[(results_df["Event_ID"] == event_id) & (results_df["Round"] == "Final")]["Athlete_ID"])


# ----------------------------------------------------------------------------
# Now run across each of the 2026 events to get a comparison

n_trials = 1000
event_results = []

for event_id in events_2026:
  sf_boulders = get_real_boulders(event_id, "Semifinal")
  f_boulders = get_real_boulders(event_id, "Final")

  # New tallies for each event to compare
  finalists_counts = {a : 0 for a in roster}
  podium_counts = {a : 0 for a in roster}
  gold_counts = {a : 0 for a in roster}

  for _ in range(n_trials):
    finalists, podium = one_trial_real_boulders(roster, sf_boulders, f_boulders, affinity, n_finalists = 8, n_podium = 3)

    for a in finalists:
      finalists_counts[a] += 1

    for a in podium:
      podium_counts[a] += 1
    if podium:
      gold_counts[podium[0]] += 1

  # Display the results in a table

  sim_table = pd.DataFrame({
      "Athlete_ID": roster,
      "Pct_Reached_Final": [finalists_counts[a] / n_trials for a in roster],
      "Pct_Podium": [podium_counts[a] / n_trials for a in roster],
      "Pct_Gold": [gold_counts[a] / n_trials for a in roster],
  }).sort_values("Pct_Podium", ascending = False)

  # The predicted podium
  predicted_podium = sim_table.head(3)["Athlete_ID"].tolist()

  # The actual podium
  actual_podium = podium_for_event(event_id, BoulderResults)

  # Obtain how accurate we've been
  overlap = set(predicted_podium) & set(actual_podium)

  event_results.append({
      "Event_ID" : event_id,
      "Predicted_Podium" : predicted_podium,
      "Actual Podium" : actual_podium,
      "Overlap" : len(overlap),
  })

  # Print the results
  print(f"\n {event_id}")
  print(sim_table.head(8).to_string(index = False))
  print(f"Predicted Podium: {predicted_podium}")
  print(f"Actual Podium: {actual_podium}")
  print(f"Overlap: {len(overlap)}")


# Print overall results across all events in 2026
EventBacktestdf = pd.DataFrame(event_results)
print("\n\n Summary across finished 2026 Events:")
print(EventBacktestdf.to_string(index = False))


# The next part is checking overlap of finalists predictions


In [ ]:
finalist_uncertainty_rows = []

for event_id in events_2026:
    sf_boulders = get_real_boulders(event_id, "Semifinal")
    f_boulders = get_real_boulders(event_id, "Final")

    finalists_counts = {a: 0 for a in roster}

    # Run many noisy trials, the previous comparison we had not included
    # the potential variation of athletes performance.
    # So in this section I include this to see if accuracy improves.
    for _ in range(n_trials):
        finalists, _ = one_trial_real_boulders(roster, sf_boulders, f_boulders, affinity)
        for a in finalists:
            finalists_counts[a] += 1

    # Convert tallies into probabilities, then take the 8 most probable finalists
    pct_reached_final = pd.DataFrame({
        "Athlete_ID": roster,
        "Pct_Reached_Final": [finalists_counts[a] / n_trials for a in roster],
    }).sort_values("Pct_Reached_Final", ascending=False)

    predicted_finalists = pct_reached_final.head(8)["Athlete_ID"].tolist()
    actual = actual_finalists_for_event(event_id, BoulderResults)
    overlap = set(predicted_finalists) & actual

    finalist_uncertainty_rows.append({
        "Event_ID": event_id,
        "Overlap": len(overlap),
        "Precision": round(len(overlap) / len(predicted_finalists), 3),
        "Recall": round(len(overlap) / len(actual), 3) if actual else None,
    })

FinalistUncertaintyDf = pd.DataFrame(finalist_uncertainty_rows)
print(FinalistUncertaintyDf.to_string(index=False))

total_overlap = FinalistUncertaintyDf["Overlap"].sum()
print("\nOverall precision (with uncertainty):", round(total_overlap / (8 * 5), 3))
print("Overall recall (with uncertainty):", round(total_overlap / (8 * 5), 3))